# Geospatial Modeling, Machine Learning, and an LLM Walk into a Bar: Mapping Stream Banks from Lidar

*GIS Week workshop*

In this hands-on workshop we use one small geospatial problem, finding the edges of a stream channel in a high-resolution
lidar terrain model, to explore how classic geospatial modeling, machine learning, and generative AI fit together.
Working with a stream in nearby Chatham County, we build the workflow with terrain analysis, flow routing, and least-cost paths,
then use its results to train a random forest and look at whether it can improve on them. Along the way, we try out Gemini,
the LLM built into Colab, on parts of the workflow and see what it is good for. Everything runs in Google Colab with GRASS and Python.
Some familiarity with Python is helpful but not required.

## The problem

Bear Creek in Chatham County, about 40 minutes from campus, is covered by a high-resolution lidar terrain model.
The lidar shows the channel clearly, while the official stream line from the National Hydrography Dataset is off by several meters in places.
Where exactly are the banks? How wide is the channel, how high are the banks, and how does that change along the 21 km of the stream?
Nobody is going to digitize that by hand, and there is no "find banks" button in any GIS. So we build one.

In part 1 we derive the stream from the terrain, describe the shape of the terrain with curvature, and let a least-cost path trace
the bank edges. In part 2 we use those results to train a random forest that predicts where banks are, and run the same path tracing
on its predictions.

Colab has Gemini built in: a chat panel that sees your notebook, code generation from a prompt in a cell, and an "explain error" button
under tracebacks. The boxes marked 🤖 are the places where we hand it something along the way.

We work in [GRASS](https://grass.osgeo.org/) through its Python API, in a Jupyter notebook on Google Colab.
GRASS manages the data and provides the few hundred tools we pick from. Python glues them together and makes the plots.

## Setup

### Install GRASS

Colab does not come with GRASS, so we install it from conda-forge. This takes a few minutes and, once, restarts the runtime.

> The next cell restarts the Colab runtime. Colab will say "Your session crashed for an unknown reason". That is expected
> and happens exactly once. Wait a few seconds for it to reconnect, then run the next cell. If you use "Run all", run it again
> after the restart; the install cell sees the existing conda and returns right away.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the restart, check that conda is ready and install GRASS, plus scikit-learn for part 2. The install is the few-minute part.

In [ ]:
import condacolab

condacolab.check()

In [ ]:
!mamba install -y -q grass=8.5 scikit-learn

In [ ]:
!grass --version

### Import the Python packages

GRASS tells us where its Python package lives (`grass --config python_path`). We add that to the Python path and import three things:
`grass.script` for session management, `grass.jupyter` for maps in the notebook, and `grass.tools`, which turns every GRASS tool
into a Python function. `tools.r_watershed(...)` runs `r.watershed`, and so on.

In [ ]:
import io
import os
import sys
import subprocess
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)
import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### One Colab fix

Colab presets the `PROJ_LIB` and `GDAL_DATA` environment variables to an older system install, which confuses the PROJ and GDAL
that came with conda. We point them at the conda files before GRASS starts. Outside Colab the cell finds no such directory and does nothing.


In [ ]:
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)


### Get the data

Everything we need is packed in one GRASS *project*, a 1 GB zip file.
A project is a directory that holds all data of an analysis in one coordinate reference system. GRASS reads and writes its maps
there, so you never deal with files during the analysis. The download takes a couple of minutes. While it runs, here is what is inside:

| Map | What it is | Where it came from |
|---|---|---|
| `dtm` | bare-earth lidar DTM, 3.125 ft (0.95 m) cells, 140 million of them | NC Emergency Management, 2024 lidar, [sdd.nc.gov](https://sdd.nc.gov/) |
| `skyview` | sky-view factor of the DTM | derived with the r.skyview addon |
| `skyview_relief` | hillshade blended with the sky-view factor, the background of the detail maps (slow to compute, so precomputed) | derived with r.relief and r.blend |
| `ortho` | aerial imagery, coarsened to 30 ft | USGS NAIP |
| `basin` | the Headwaters Bear Creek watershed, 66 km² | USGS Watershed Boundary Dataset |
| `bear_creek` | the official stream line | USGS National Hydrography Dataset |

The project is in North Carolina State Plane with US survey feet as units, because that is how the state publishes its lidar.
We do the GIS work in feet and switch to meters for the plots.
The full dataset and the notebook that built it from the original sources are linked at the end.

<details>
<summary>How the dataset was built</summary>

The project was created from the DTM with `gs.create_project`, so it inherits the CRS of the GeoTIFF, EPSG:6543.

- `dtm`: bare-earth DEM from the 2024 QL1 lidar of Chatham County, distributed as a countywide GeoTIFF at 3.125 ft (0.95 m) by the
  NC Emergency Management [Spatial Data Download](https://sdd.nc.gov/) portal. Imported with `r.import` for the extent of the watershed
  grown by 1000 ft, 10 307 rows by 13 641 columns.
- `basin`: hydrologic unit 030300030507, Headwaters Bear Creek, 66 km², from the USGS Watershed Boundary Dataset map service
  (`v.in.ags` addon).
- `bear_creek`: the NHD flowlines named Bear Creek from the USGS National Hydrography Dataset map service, clipped to the watershed
  with `v.clip`.
- `ortho`: NAIP imagery from the USGS imagery service, fetched with `r.in.wms` for the DTM extent at 30 ft.
- `skyview`: sky-view factor of the DTM, `r.skyview` addon with a 100 ft horizon search distance.
- `skyview_relief`: `r.blend -c` of a hillshade from `r.relief` and the sky-view factor, half and half.

```python
gs.create_project(project, filename=dtm_file)
session = gj.init(project)
tools = Tools()

tools.g_extension(extension="v.in.ags")
tools.v_in_ags(url="https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/", where="huc12 = '030300030507'", output="basin")
bbox = tools.g_region(vector="basin", flags="b", format="json")
tools.v_in_ags(url="https://hydro.nationalmap.gov/arcgis/rest/services/nhd/MapServer/6/", where="gnis_name = 'Bear Creek'",
               bbox_filter=f"{bbox['ll_w']},{bbox['ll_s']},{bbox['ll_e']},{bbox['ll_n']}", output="creek")
tools.v_clip(input="creek", clip="basin", output="bear_creek")

tools.g_region(vector="basin", res=1, grow=1000)
tools.r_import(input=dtm_file, output="dtm", extent="region")

tools.g_region(raster="dtm", res=30)
tools.r_in_wms(url="https://imagery.nationalmap.gov/arcgis/services/USGSNAIPImagery/ImageServer/WMSServer", out="ortho", layer="USGSNAIPImagery")

tools.g_region(raster="dtm")
tools.g_extension(extension="r.skyview")
tools.r_skyview(input="dtm", output="skyview", maxdistance=100)
tools.r_relief(input="dtm", output="relief")
tools.r_blend(flags="c", first="relief", second="skyview", output="skyview_relief")
```
</details>


In [ ]:
data_dir = Path("/content")  # change this if you run locally
project = data_dir / "chatham_northcarolina_usa_epsg6543"

url = "https://github.com/ncsu-geoforall-lab/gis-week-2026/releases/download/v1/chatham_northcarolina_usa_epsg6543_workshop.zip"
archive, _ = urllib.request.urlretrieve(url)
with zipfile.ZipFile(archive) as zip_ref:
    zip_ref.extractall(data_dir)
Path(archive).unlink()


### Start GRASS

`gj.init` starts a GRASS session in the project. The `tools` object is how we call GRASS tools for the rest of the notebook.
As a first call, we ask `g.proj` for the name of the coordinate reference system.


In [ ]:
session = gj.init(project)
tools = Tools()
print("GRASS", gs.version()["version"])
tools.g_proj(flags="p", format="projjson")["name"]


### A first look

Before computing anything, let's look at the area on a web map. The backgrounds come from web services: the state's 3 ft lidar hillshade
from NC OneMap and USGS aerial imagery. `gj.Vector` reprojects a GRASS vector map to latitude and longitude and drops it onto a
[folium](https://python-visualization.github.io/folium/) map, here the watershed and the official stream line.
The map setup goes into a function because we come back to it at the end of part 1.
The imagery is on top; uncheck it in the layer control to see the hillshade, and zoom in on the stream.


In [ ]:
import folium


def web_map(vector):
    """A folium map zoomed to a GRASS vector map, with lidar hillshade and imagery as backgrounds."""
    # query the extent in lat/lon without changing the computational region
    with gs.RegionManager(vector=vector):
        bounds = tools.g_region(flags="b", format="json")
    web = folium.Map(tiles="Esri.WorldGrayCanvas")
    web.fit_bounds([[bounds["ll_s"], bounds["ll_w"]], [bounds["ll_n"], bounds["ll_e"]]])
    folium.WmsTileLayer(
        url="https://services.nconemap.gov/secure/services/Elevation/DEM03_hillshade/ImageServer/WMSServer",
        layers="DEM03_hillshade:hillshade",
        fmt="image/png",
        transparent=True,
        version="1.3.0",
        attr="NC OneMap, NC Emergency Management",
        name="Lidar hillshade (3 ft)",
    ).add_to(web)
    folium.WmsTileLayer(
        url="https://imagery.nationalmap.gov/arcgis/services/USGSNAIPImagery/ImageServer/WMSServer",
        layers="USGSNAIPImagery",
        fmt="image/png",
        transparent=True,
        version="1.3.0",
        attr="USGS National Agriculture Imagery Program (NAIP)",
        name="NAIP imagery",
    ).add_to(web)
    return web


first_look = web_map("basin")
gj.Vector("basin", style_function=lambda feature: {"color": "black", "fill": False, "weight": 3}).add_to(first_look)
gj.Vector("bear_creek", style_function=lambda feature: {"color": "#77caf9", "weight": 3}).add_to(first_look)
folium.LayerControl().add_to(first_look)
first_look


### Rendering with GRASS

The web map is good for looking around. The maps in the rest of the notebook are rendered by GRASS itself.
`gj.Map` collects display commands, the same `d.*` tools as in the GRASS GUI, and renders them to an image.
The overview: the orthophoto, the watershed outline and the official stream line.


In [ ]:
overview = gj.Map(width=1000)
overview.d_rast(map="ortho")
overview.d_vect(map="basin", width=4, color="black", fill_color="none")
overview.d_vect(map="bear_creek", width=4, color="#77caf9")
overview.d_barscale(style="solid", at=[1, 5], units="kilometers", length=2, fontsize=20, bgcolor="none")
overview.show()

Every raster operation and every map in GRASS runs within the *computational region*: an extent and a resolution
that you set once and all tools respect. Two *named regions* store the extents of the detail reaches we keep coming back to:
reach A in the middle of the stream, and reach B at the outlet. `RegionManager` switches to a saved region
for the duration of a `with` block and switches back afterwards.


In [ ]:
tools.g_region(n=676500, s=673500, e=1885900, w=1881900, save="zoom_A")
tools.g_region(n=677200, s=675700, e=1888000, w=1885900, save="zoom_B")

Reach A up close, with the DTM over `skyview_relief`, a hillshade blended with the sky-view factor. The blend shows the incised channel
much better than a hillshade alone, and it is the background of all detail maps below.
Look at the channel bottom: part of it is perfectly flat, part of it is a mess of triangles.
Lidar does not see through water, so the DTM in the channel is either a flattened water surface or an interpolation across a few
stray points. That is fine for the banks, which are above the water, but keep it in mind whenever we talk about the channel bottom.


In [ ]:
with gs.RegionManager(region="zoom_A"):
    detail = gj.Map(use_region=True, width=1000)
    detail.d_shade(color="dtm", shade="skyview_relief")
    detail.d_vect(map="bear_creek", width=3, color="#77caf9", legend_label="Bear Creek (NHD)")
    detail.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=20, bgcolor="none")
    detail.show()

Reach B, the last stretch before the stream leaves the watershed.


In [ ]:
with gs.RegionManager(region="zoom_B"):
    detail = gj.Map(use_region=True, width=1000)
    detail.d_shade(color="dtm", shade="skyview_relief")
    detail.d_vect(map="bear_creek", width=3, color="#77caf9", legend_label="Bear Creek (NHD)")
    detail.d_barscale(style="solid", at=[2, 5], units="meters", length=100, fontsize=20, bgcolor="none")
    detail.show()

## Part 1: Geospatial modeling

### Step 1: Find the stream

The bank tracing needs a stream line that sits inside the channel, and the official line does not, so we derive our own from the terrain.
`r.watershed` does flow routing on the whole DTM in one pass: for every cell, which way does water go, how many cells drain through it,
and where does that count get large enough to call it a stream. We start a stream at 50 000 cells, about 4.5 ha of drainage.
The `b` flag routes the flow near the middle of the flat, hydro-flattened sections of the channel instead of along their edges. The `a` flag keeps the accumulation positive everywhere; without it, cells whose count may be underestimated get negative values.

The first cell sets the computational region to the whole DTM. The second is the slow one, about 4 minutes.

> 🤖 While it runs, select the cell and ask Gemini to explain it. Then open the
> [r.watershed manual](https://grass.osgeo.org/grass-stable/manuals/r.watershed.html) and check two things: is `threshold`
> in cells or in area units, and what does the `b` flag do? The explanation reads fluently either way. The manual is where you find
> out whether it is right.

In [ ]:
tools.g_region(raster="dtm")

In [ ]:
tools.r_watershed(
    elevation="dtm",
    accumulation="accumulation",
    flags="ab",
    threshold=50000,
    drainage="direction",
    stream="streams",
)

The flow accumulation in reach A: how many cells drain through each cell, on a log scale. The line of high values is the
derived stream, and it runs inside the channel.


In [ ]:
with gs.RegionManager(region="zoom_A"):
    accum = gj.Map(use_region=True, width=1000)
    accum.d_shade(color="accumulation", shade="skyview_relief")
    accum.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    accum.d_legend(raster="accumulation", title="Accumulation (cells)",  flags="tl", fontsize=14, at=[5, 10, 60, 95])
    accum.show()

That is the whole network, thousands of segments, and we want Bear Creek, the main stem. Stream ordering does that for us.
Hack's order gives 1 to the main stem of every network, traced from the outlet to the most distant source. The `r.stream.order`
addon computes it (addons are installed on the fly with `g.extension`). We rasterize the watershed polygon and use it as a mask,
so the network stops at the watershed boundary instead of running on to the edge of the DTM. `MaskManager` applies the mask
only inside the `with` block.


In [ ]:
tools.g_extension(extension="r.stream.order")


In [ ]:
tools.v_to_rast(input="basin", output="basin_mask", use="value")
with gs.MaskManager("basin_mask"):
    tools.r_stream_order(
        stream_rast="streams",
        accumulation="accumulation",
        direction="direction",
        elevation="dtm",
        hack="stream_hack",
        stream_vect="stream_vector_hack",
    )


The network as a vector map, with the order 1 segments highlighted in blue.


In [ ]:
network = gj.Map(width=1000)
network.d_rast(map="skyview_relief")
network.d_vect(map="stream_vector_hack", type="line", color="grey", width=1)
network.d_vect(map="stream_vector_hack", type="line", where="hack = 1", color="aqua", width=3)
network.d_barscale(style="solid", at=[1, 5], units="kilometers", length=2, fontsize=20, bgcolor="none")
network.show()

We pull out the order 1 segments and merge them into continuous lines.

In [ ]:
tools.v_extract(input="stream_vector_hack", where="hack = 1", output="hack_1", type="line")
tools.v_build_polylines(input="hack_1", output="connected_hack_1", cats="first")

A few short order 1 fragments survive at the watershed boundary, where neighboring networks poke in. Bear Creek is the longest one,
so we ask for the lengths as JSON, drop them into a pandas DataFrame and keep the longest. This JSON-to-pandas move is the standard way
to get numbers out of GRASS and into Python.

In [ ]:
lengths = pd.DataFrame(
    tools.v_to_db(map="connected_hack_1", option="length", type="line", flags="p", format="json")["records"]
)
longest_cat = int(lengths.loc[lengths["length"].idxmax(), "category"])
tools.v_extract(input="connected_hack_1", output="longest_hack_1", type="line", cat=longest_cat)
lengths.sort_values("length", ascending=False)

> 🤖 Ask Gemini to write a cell that computes the length of every line in `connected_hack_1` and prints the longest.
> It will most likely use `grass.script.run_command` or `read_command`, the API from the GRASS tutorials and Stack Exchange answers of the
> last fifteen years. That still works, but it is not the `tools` object we use here. Paste one `tools.` call from this notebook
> into the prompt as an example and ask again. An LLM knows the APIs that were common when its training data was written.
> For a new one, you have to show it.

A line traced through raster cells zigzags at the cell scale, and the zigzag is worst over the faceted parts of the channel bottom.
It would throw off every distance we measure later. Two passes of `v.generalize` fix that: simplify with a 10 ft tolerance, then
round the corners. The result, `stream`, is our centerline.

In [ ]:
tools.v_generalize(input="longest_hack_1", method="reumann", threshold=10, output="reumann")
tools.v_generalize(input="reumann", method="chaiken", threshold=10, output="stream")

Compare it with the official line in reach A.

In [ ]:
with gs.RegionManager(region="zoom_A"):
    compare = gj.Map(use_region=True, width=1000)
    compare.d_shade(color="dtm", shade="skyview_relief")
    compare.d_vect(map="bear_creek", color="blue", width=3, legend_label="NHD")
    compare.d_vect(map="stream", color="#77caf9", width=3, legend_label="Derived from DTM")
    compare.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    compare.show()

### Step 2: Describe the terrain

Now the main idea of this workshop. A bank is a convex break in slope, an edge. Imagine walking along the stream:
if walking on an edge were cheap and walking anywhere else expensive, the cheapest route from one end of the stream to the other
would follow the bank. That is a least-cost path problem, and GRASS solves those. We need two ingredients: a measure of "edginess"
and a corridor, so we do not search the whole county.

The corridor is a 70 ft (21 m) buffer on each side of the centerline, wide enough for the widest parts of the channel.
Edginess is *maximum curvature*, computed by `r.param.scale`, which fits a surface to a 7 by 7 cell window (about 7 m) around each cell.
Convex edges such as bank tops have high positive curvature. About a minute.

<details>
<summary>Why zscale=3.28084?</summary>

Curvature has units of 1/length, so its value depends on the length unit. Our project is in feet, which makes curvature values 3.28 times
smaller than they would be in meters. Scaling the elevations by 3.28084 ft/m makes the values match a metric project, so the threshold
we use below would work unchanged on metric data.
</details>

> 🤖 Error drill. Before running the next cell as written, change `method="maxic"` to `method="maximum"` and run it.
> GRASS refuses, with the list of valid values. Click "Explain error" or paste the traceback to Gemini and compare its
> explanation with the error message itself. Then put `maxic` back. Where the error message is good, the LLM adds little.
> It earns its keep on the cryptic ones, and on the ones three layers down a traceback.

In [ ]:
tools.v_buffer(input="stream", output="buffer", type="line", distance=70, flags="c")
tools.r_param_scale(input="dtm", output="maxcurvature", size=7, method="maxic", zscale=3.28084)

In [ ]:
with gs.RegionManager(region="zoom_A"):
    curvature = gj.Map(use_region=True, width=1000)
    curvature.d_shade(color="maxcurvature", shade="skyview_relief")
    curvature.d_vect(map="buffer", color="200:200:200", width=2, fill_color="none", legend_label="Corridor")
    curvature.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    curvature.d_legend(raster="maxcurvature", title="Curvature (1/m)", label_step=0.1, flags="t", fontsize=14, at=[5, 10, 60, 95])
    curvature.show()

The bank edges show up as continuous lines of high curvature. So do road embankments, ditches and field terraces, which is why the
corridor matters.

### Step 3: Build the cost surface

Cost surface number one is made from curvature with raster algebra. Cells with curvature above 0.04 get a cost below 1 (one minus
the curvature), everything else in the corridor costs 100, and the centerline itself costs 100 000. That last one makes the stream
a wall: a path that starts on the left bank can never cross to the right. The exact numbers are empirical; only their ratios matter.

We do this inside a mask set to the corridor, so the cost surface exists only there. The wall is the centerline buffered by one cell
on each side and rasterized as an area: three cells wide, with no gaps. A rasterized line is one cell wide and can have gaps where
it bends sharply, and a path slips through a gap as soon as the other side is cheaper. In part 2, a random forest gives us cost
surface number two.

In [ ]:
baseline_cost = 100
stream_cost = 100000
curvature_threshold = 0.04

tools.v_buffer(input="stream", output="wall", type="line", distance=3.125)
with gs.MaskManager():
    tools.r_mask(vector="buffer")
    tools.v_to_rast(input="wall", output="wall", use="val")
    tools.r_mapcalc(
        expression=(
            f"cost_curvature = if(isnull(wall), "
            f"if(maxcurvature > {curvature_threshold}, 1 - maxcurvature, {baseline_cost}), "
            f"{stream_cost})"
        )
    )

> 🤖 Ask Gemini: *In a least-cost path on this cost surface, why does the stream centerline cost 100 000 and not
> just 100? What changes if the bank cells cost 50 instead of less than 1?* Conceptual questions like this are where a general model
> is on firm ground. It does not need to know GRASS to answer them.

### Step 4: Trace the banks

A least-cost path needs a start and an end. `v.segment` places points along a line at a given distance and a sideways offset:
we put one point on each side of the centerline, 15 ft out, 10 ft from each end of the stream. The centerline runs downstream,
so the start points are at the outlet and the end points at the headwaters. A positive side offset is the right side, looking downstream.
The rules are passed to the tool as text.

In [ ]:
offset = 10
side_offset = 15

start_rules = f"P 1 {longest_cat} -{offset} {side_offset}\nP 2 {longest_cat} -{offset} -{side_offset}"
end_rules = f"P 1 {longest_cat} {offset} {side_offset}\nP 2 {longest_cat} {offset} -{side_offset}"
tools.v_segment(input="stream", output="start_points", rules=io.StringIO(start_rules))
tools.v_segment(input="stream", output="end_points", rules=io.StringIO(end_rules))

The points on an interactive map. `gj.InteractiveMap` is the quick way to a web map: it reprojects the vector maps itself
and adds a layer control. Zoom in on the two ends of the stream to find the points.


In [ ]:
point_map = gj.InteractiveMap(tiles="Esri.WorldTopoMap")
point_map.add_vector(name="start_points")
point_map.add_vector(name="end_points")
point_map.add_vector(name="stream")
point_map.show()

Reach B at the outlet with the cost surface: yellow is cheap (the bank edges), green is the baseline, purple is the wall,
and the black pins are the start points.

In [ ]:
cost_colors = "0 #F2E51E\n1 #F2E51E\n100 #209F88\n100000 #482072"
tools.r_colors(map="cost_curvature", rules=io.StringIO(cost_colors))

with gs.RegionManager(region="zoom_B"):
    cost_map = gj.Map(use_region=True, width=1000)
    cost_map.d_shade(color="cost_curvature", shade="skyview_relief")
    cost_map.d_vect(map="start_points", size=10, icon="basic/pin", color="black", fill_color="black", legend_label="Start points")
    cost_map.d_barscale(style="solid", at=[2, 5], units="meters", length=100, fontsize=20, bgcolor="none")
    cost_map.d_legend(raster="cost_curvature", title="Cost", label_values=[1, 100, 100000], flags="tl", fontsize=14, at=[5, 9, 60, 95])
    cost_map.show()

Two tools do the tracing. `r.cost` computes, from the start points, the cumulative cost of reaching every cell in the corridor,
and records the direction each cell was reached from. The `k` flag allows knight's moves, which makes the paths less jagged. A knight's move could jump a one-cell wall; it cannot
jump three cells.
About half a minute.

In [ ]:
tools.r_cost(flags="k", input="cost_curvature", output="cumulative_cost", outdir="cost_direction", start_points="start_points")

The cumulative cost in reach B. It is lowest at the start points and grows upstream, slowly along the bank edges and fast anywhere else,
so the colors are on a log scale.

In [ ]:
tools.r_colors(map="cumulative_cost", color="viridis", flags="g")

with gs.RegionManager(region="zoom_B"):
    stats = tools.r_univar(map="cumulative_cost", format="json")
    cumulative = gj.Map(use_region=True, width=1000)
    cumulative.d_shade(color="cumulative_cost", shade="skyview_relief")
    cumulative.d_vect(map="start_points", size=10, icon="basic/pin", color="black", fill_color="black", legend_label="Start points")
    cumulative.d_barscale(style="solid", at=[2, 5], units="meters", length=100, fontsize=20, bgcolor="none")
    cumulative.d_legend(raster="cumulative_cost", title="Cumulative cost", range=[1, stats["max"]], label_values=[1, 10, 100, 1000, 10000], flags="tl", fontsize=14, at=[5, 9, 60, 95])
    cumulative.show()

`r.path` then walks from each end point back downhill along the recorded directions to the start points. Because of the wall,
each path stays on its side. The result is a vector map with two lines, category 1 for the right bank and category 2 for the left bank.

In [ ]:
tools.r_path(input="cost_direction", vector_path="banks", start_points="end_points")

with gs.RegionManager(region="zoom_B"):
    banks_map = gj.Map(use_region=True, width=1000)
    banks_map.d_rast(map="skyview_relief")
    banks_map.d_vect(map="banks", width=2, color="black")
    banks_map.d_vect(map="stream", width=1, color="aqua")
    banks_map.d_barscale(style="solid", at=[2, 5], units="meters", length=100, fontsize=20, bgcolor="none")
    banks_map.show()

Now the banks along the whole stream, on the web map from the first look, this time zoomed to the stream.
The imagery is on top; uncheck it in the layer control to see the hillshade. Zoom in: the banks are only meaningful at the scale of the channel.


In [ ]:
banks_web = web_map("stream")
gj.Vector("stream").add_to(banks_web)
gj.Vector("banks", style_function=lambda feature: {"color": "black"}).add_to(banks_web)
folium.LayerControl().add_to(banks_web)
banks_web

### Step 5: Measure the channel

Two lines are nice to look at, but the point was numbers: how wide, how high, where. For bank height we use HAND, Height Above
Nearest Drainage: for every cell, the elevation difference to the stream cell that water from it would flow to.
The `r.stream.distance` addon computes it from the stream raster and the flow directions we already have. About a minute and a half.

In [ ]:
tools.g_extension(extension="r.stream.distance")
tools.r_stream_distance(stream_rast="streams", direction="direction", elevation="dtm", difference="hand")

In [ ]:
tools.r_colors(map="hand", rules=io.StringIO("0 #440154\n2.5 #3b528b\n5 #21918c\n7.5 #5ec962\n10 #fde725\n1000 #fde725"))

with gs.RegionManager(region="zoom_A"):
    hand_map = gj.Map(use_region=True, width=1000)
    hand_map.d_shade(color="hand", shade="skyview_relief")
    hand_map.d_vect(map="banks", width=2, color="yellow", legend_label="Banks")
    hand_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    hand_map.d_legend(raster="hand", title="HAND (ft)", range=[0, 10], label_step=2, flags="t", fontsize=14, at=[5, 10, 60, 95])
    hand_map.show()

Now we turn the two bank lines into points every 30 ft and sample HAND and the elevation at each point. `v.to.points` writes the points
into a second attribute layer of the map, with their distance along the line in a column called `along`. That is the distance along
the bank, and the two banks have different lengths, so we replace it with the distance along the centerline: `v.distance` finds the
nearest point on the centerline for every bank point and, with `to_along`, uploads how far along the centerline that point is.
The samples go into the same layer, so every `v.what.rast` call says `layer=2`.


In [ ]:
spacing = 30  # ft

tools.v_extract(input="banks", cats="1", output="right_bank")
tools.v_extract(input="banks", cats="2", output="left_bank")
for side in ("right_bank", "left_bank"):
    tools.v_to_points(input=side, output=f"{side}_points", dmax=spacing)
    tools.v_distance(from_=f"{side}_points", from_layer=2, to="stream", upload="to_along", column="along")
    tools.v_what_rast(map=f"{side}_points", raster="hand", column="hand", layer=2)
    tools.v_what_rast(map=f"{side}_points", raster="dtm", column="dtm", layer=2)

The right bank points in reach A, colored by their HAND value. `v.colors` writes a color for each point into the attribute table,
and `d.vect` reads it from there.

In [ ]:
tools.v_colors(map="right_bank_points", layer=2, use="attr", column="hand", range=[0, 10], color="viridis", rgb_column="GRASSRGB")

with gs.RegionManager(region="zoom_A"):
    points_map = gj.Map(use_region=True, width=1000)
    points_map.d_rast(map="skyview_relief")
    points_map.d_vect(map="left_bank_points", layer=2, icon="basic/circle", size=5, color="none", fill_color="200:200:200")
    points_map.d_vect(map="right_bank_points", layer=2, icon="basic/circle", size=7, color="none", rgb_column="GRASSRGB")
    points_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    points_map.d_legend(raster="hand", title="HAND of right bank (ft)", range=[0, 10], label_step=2, flags="t", fontsize=14, at=[5, 10, 60, 95])
    points_map.show()

The width: `v.distance` pairs every right bank point with its nearest left bank point, uploads the distance and the HAND and elevation
of that point into the right bank table, and can also draw the pairs as lines.

In [ ]:
tools.v_db_addcolumn(
    map="right_bank_points",
    layer=2,
    column="width double precision,hand_left double precision,dtm_left double precision",
)
tools.v_distance(
    from_="right_bank_points", from_layer=2, to="left_bank_points", to_layer=2,
    upload="dist,to_attr", to_column="hand", column="width,hand_left", output="width_lines",
)
tools.v_distance(
    from_="right_bank_points", from_layer=2, to="left_bank_points", to_layer=2,
    upload="to_attr", to_column="dtm", column="dtm_left",
)

In [ ]:
with gs.RegionManager(region="zoom_A"):
    width_map = gj.Map(use_region=True, width=1000)
    width_map.d_rast(map="skyview_relief")
    width_map.d_vect(map="banks", width=2, color="black", legend_label="Banks")
    width_map.d_vect(map="width_lines", width=1, color="red", legend_label="Width")
    width_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    width_map.show()

The table now has everything we need, in feet. We read it into pandas, the same JSON move as before.

In [ ]:
profile = pd.DataFrame(
    tools.v_db_select(
        map="right_bank_points", layer=2, columns="along,hand,dtm,width,hand_left,dtm_left", format="json"
    )["records"]
)
profile = profile.apply(pd.to_numeric, errors="coerce").sort_values("along").reset_index(drop=True)
profile = profile.rename(columns={"hand": "hand_right", "dtm": "dtm_right"})
profile.head()

Then meters, kilometers along the stream, and a few derived columns: the lower, mean and higher of the two bank heights, their
difference, and the channel bottom as bank elevation minus HAND, averaged over both banks.

<details>
<summary>Why bank minus HAND instead of sampling the DTM along the centerline?</summary>

HAND is the difference between a bank cell and the stream cell it drains to, so bank elevation minus HAND *is* the elevation of a stream
cell. Doing it this way keeps bank top, bank height and channel bottom tied to the same point and the same distance along the bank,
with no need to match stations between the centerline and the banks.
</details>

In [ ]:
FT_TO_M = 0.3048

profile = profile * FT_TO_M
profile["along"] /= 1000  # km
profile["bottom"] = ((profile.dtm_right - profile.hand_right) + (profile.dtm_left - profile.hand_left)) / 2
profile["hand_low"] = profile[["hand_right", "hand_left"]].min(axis=1)
profile["hand_mean"] = profile[["hand_right", "hand_left"]].mean(axis=1)
profile["hand_high"] = profile[["hand_right", "hand_left"]].max(axis=1)
profile["hand_diff"] = profile.hand_right - profile.hand_left
profile.describe().round(2)

Now the plots, all against the distance along the stream. The faint line is the raw value every 9 m, the solid one a 30-point
(about 270 m) rolling mean. First the width.

In [ ]:
from matplotlib.ticker import MultipleLocator

SMOOTH = 30  # rolling window in points, about 270 m


def smooth(series):
    return series.rolling(SMOOTH, center=True, min_periods=1).mean()


def profile_axes(ylabel):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.set_xlabel("Distance along stream (km)")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.grid(alpha=0.3)
    ax.margins(x=0.01)
    return ax


ax = profile_axes("Width (m)")
ax.plot(profile.along, profile.width, "k", alpha=0.25, lw=1)
ax.plot(profile.along, smooth(profile.width), "k", lw=2);

The bank heights: the higher and the lower of the two banks and their mean. The lower one decides when the stream overtops.

In [ ]:
C_HIGH, C_MEAN, C_LOW = "#D01C8B", "#2F3538", "#4DAC26"

ax = profile_axes("HAND (m)")
for column, color, label in (("hand_high", C_HIGH, "high bank"), ("hand_mean", C_MEAN, "mean bank"), ("hand_low", C_LOW, "low bank")):
    ax.plot(profile.along, profile[column], color=color, alpha=0.25, lw=1)
    ax.plot(profile.along, smooth(profile[column]), color=color, lw=2, label=label)
ax.legend(loc="upper left");

Which bank is higher: the difference between the right and the left bank height.

In [ ]:
ax = profile_axes("Right − left HAND (m)")
diff = smooth(profile.hand_diff)
ax.fill_between(profile.along, 0, diff, where=diff >= 0, color=C_HIGH, alpha=0.5, label="right bank higher")
ax.fill_between(profile.along, 0, diff, where=diff < 0, color=C_LOW, alpha=0.5, label="left bank higher")
ax.axhline(0, color="k", lw=0.8)
ax.legend(loc="upper left");

And the channel bottom.

In [ ]:
ax = profile_axes("Channel bottom (m)")
ax.plot(profile.along, profile.bottom, "k--", lw=1.5, alpha=0.25)
ax.plot(profile.along, smooth(profile.bottom), "k--", lw=1.5);

Twenty-one kilometers of stream in four plots. Bear Creek starts about 8 m wide and ends 15 to 20 m wide. The banks grow from
half a meter to about 2 m, and the right bank is the higher one along most of the lower stream. The spikes in the width are real places:
road crossings, tributary junctions, and spots where the path lost the edge. The biggest one is between km 6 and 8, where the
channel comes out 30 m wide. We select the bank points in that range, ask for their extent, and render it.

In [ ]:
range_km = (6, 8)
range_ft = [km * 1000 / FT_TO_M for km in range_km]
zoom = tools.v_db_select(
    map="right_bank_points",
    layer=2,
    flags="r",
    format="json",
    where=f"along > {range_ft[0]} and along < {range_ft[1]}",
)["extent"]

with gs.RegionManager(grow=100, **zoom):
    widest = gj.Map(use_region=True, width=1000)
    widest.d_rast(map="skyview_relief")
    widest.d_vect(map="banks", width=2, color="black", legend_label="Banks")
    widest.d_vect(map="stream", width=2, color="aqua", legend_label="Centerline")
    widest.d_barscale(style="solid", at=[2, 5], units="meters", length=200, fontsize=14, bgcolor="none")
    widest.show()

That is part 1: a stream, two banks and a profile, from nothing but a terrain model, with about a dozen GRASS tools.
The weak spot is visible in the map. The cost surface is one hand-tuned rule on one terrain variable, and wherever the edge
is faint the path wanders. Part 2 asks whether a model trained on these results can do better.

## Part 2: Machine learning

Part 1 found the banks with one rule on one variable. A model could combine several variables and spare us the hand tuning.
The catch is that supervised learning needs examples of banks, and nobody has digitized the banks of Bear Creek for us.
What we do have are the banks from part 1: 56 000 corridor cells that the curvature path called bank, and 930 000 it did not.
We use those as labels, with one precaution. A model trained and tested on the same cells can only tell us how well it memorized them,
so we train on three short blocks, 2 km each, and judge the model on the other 15 km, which it never sees.
The blocks skip the first two kilometers of the stream, where the channel is smallest and the part 1 banks are least reliable.
The km 6 to 8 spike is in the held-out part.

> 🤖 Before running anything, ask Gemini: *I want to train a random forest to detect stream bank cells in a lidar DTM. My only
> labels are the banks that a curvature-based least-cost path found. What can go wrong?* Keep the answer for the end of part 2.

### Step 6: Features and labels

Features are the terrain variables the model gets to look at. Next to maximum curvature we add profile and minimum curvature from
the same tool, slope, HAND, and the distance from the centerline, which is a hint about where banks usually are relative to the stream.
We also shrink the computational region to the extent of the corridor, plus a few cells for the moving windows, which halves the number
of cells every tool has to process. The distance is only needed inside the corridor, so we clip it to the corridor mask.
About half a minute.

In [ ]:
tools.g_region(vector="buffer", align="dtm", grow=10)
tools.r_slope_aspect(elevation="dtm", slope="slope")
tools.r_param_scale(input="dtm", output="profcurvature", size=7, method="profc", zscale=3.28084)
tools.r_param_scale(input="dtm", output="mincurvature", size=7, method="minic", zscale=3.28084)
tools.r_grow_distance(input="wall", distance="dist_stream_all")
with gs.MaskManager():
    tools.r_mask(vector="buffer")
    tools.r_mapcalc(expression="dist_stream = dist_stream_all")

The four new features in reach A, one map each: slope, profile curvature, minimum curvature and distance from the centerline.
Each one sees the bank in its own way: slope is steep just below the bank top, profile curvature is convex on it, minimum curvature
is concave in the channel bottom, and the distance tells how far from the stream we are. The colors are stretched by histogram equalization
(flag `e` of `r.colors`), otherwise a few extreme cells would hide the differences.


In [ ]:
tools.r_colors(map="slope", color="viridis", flags="e")

with gs.RegionManager(region="zoom_A"):
    stats = tools.r_univar(map="slope", format="json")
    feature_map = gj.Map(use_region=True, width=1000)
    feature_map.d_shade(color="slope", shade="skyview_relief")
    feature_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    feature_map.d_legend(raster="slope", title="Slope (degrees)", range=[stats["min"], stats["max"]], flags="t", fontsize=14, at=[5, 10, 60, 95])
    feature_map.show()


In [ ]:
tools.r_colors(map="profcurvature", color="viridis", flags="e")

with gs.RegionManager(region="zoom_A"):
    stats = tools.r_univar(map="profcurvature", format="json")
    feature_map = gj.Map(use_region=True, width=1000)
    feature_map.d_shade(color="profcurvature", shade="skyview_relief")
    feature_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    feature_map.d_legend(raster="profcurvature", title="Profile curvature (1/m)", range=[stats["min"], stats["max"]], flags="t", fontsize=14, at=[5, 10, 60, 95])
    feature_map.show()


In [ ]:
tools.r_colors(map="mincurvature", color="viridis", flags="e")

with gs.RegionManager(region="zoom_A"):
    stats = tools.r_univar(map="mincurvature", format="json")
    feature_map = gj.Map(use_region=True, width=1000)
    feature_map.d_shade(color="mincurvature", shade="skyview_relief")
    feature_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    feature_map.d_legend(raster="mincurvature", title="Minimum curvature (1/m)", range=[stats["min"], stats["max"]], flags="t", fontsize=14, at=[5, 10, 60, 95])
    feature_map.show()


In [ ]:
tools.r_colors(map="dist_stream", color="viridis", flags="e")

with gs.RegionManager(region="zoom_A"):
    stats = tools.r_univar(map="dist_stream", format="json")
    feature_map = gj.Map(use_region=True, width=1000)
    feature_map.d_shade(color="dist_stream", shade="skyview_relief")
    feature_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    feature_map.d_legend(raster="dist_stream", title="Distance from centerline (ft)", range=[stats["min"], stats["max"]], flags="t", fontsize=14, at=[5, 10, 60, 95])
    feature_map.show()


The training blocks are three stretches of the centerline, 2 km each, starting at km 2, 8.5 and 17 from the headwaters: one in the upper,
one in the middle and one in the lower stream, with the first two kilometers left out on purpose. The stream is small there, the banks
from part 1 are the least reliable, and a model should not learn what a bank looks like from them.
`v.segment` cuts the blocks (an `L` rule gives the part of a line between two offsets along it, in map units, so feet here,
one rule per block) and `v.buffer` turns them into zones like the corridor.


In [ ]:
block_starts = (2, 8.5, 17)  # km along the stream from the headwaters
block_length = 2  # km

rules = "\n".join(
    f"L {i} {longest_cat} {start * 1000 / FT_TO_M} {(start + block_length) * 1000 / FT_TO_M}"
    for i, start in enumerate(block_starts, 1)
)
tools.v_segment(input="stream", output="training_line", rules=io.StringIO(rules))
tools.v_buffer(input="training_line", output="training_zone", distance=70, flags="c")


In [ ]:
split_map = gj.Map(width=1000)
split_map.d_rast(map="skyview_relief")
split_map.d_vect(map="basin", width=4, color="black", fill_color="none")
split_map.d_vect(map="stream", width=3, color="#555555", legend_label="Held out")
split_map.d_vect(map="training_line", width=4, color="red", legend_label="Training blocks")
split_map.d_barscale(style="solid", at=[1, 5], units="kilometers", length=2, fontsize=20, bgcolor="none")
split_map.show()


The labels are two rasters: the cells on a part 1 bank line, and the cells in the training blocks. Both are made inside the corridor mask,
so cells outside the lines and blocks come out as no data. `r.null` turns those into zeros, which gives the table a clean 0/1 column.


In [ ]:
with gs.MaskManager():
    tools.r_mask(vector="buffer")
    tools.v_to_rast(input="banks", output="bank_cells", use="val", flags="d")
    tools.v_to_rast(input="training_zone", output="training_cells", use="val")
tools.r_null(map="bank_cells", null=0)
tools.r_null(map="training_cells", null=0)


Now a table with one row per corridor cell: the features, the bank flag and the training flag. `r.stats` prints the values of several
rasters side by side (`-1` one cell per line, `-g` with coordinates, `-N` skipping cells that have no data in all maps), and because
the mask is set to the corridor only corridor cells come out. pandas reads the text. This is the raster-to-table move, the counterpart
of the JSON-to-pandas move for vector attributes. About a minute.

In [ ]:
features = ["maxcurvature", "profcurvature", "mincurvature", "slope", "hand", "dist_stream"]

with gs.MaskManager():
    tools.r_mask(vector="buffer")
    table = tools.r_stats(flags="1gN", input=features + ["bank_cells", "training_cells"], separator="comma")

cells = pd.read_csv(io.StringIO(table.text), names=["x", "y"] + features + ["is_bank", "training"], na_values="*")
cells.head()

A quick check of the table: the number of cells, how many of them are on a bank line, and the share in the training blocks.
About one cell in twenty is a bank cell, which is why the classes get balanced in the next step.


In [ ]:
print(f"{len(cells)} corridor cells, {cells.is_bank.sum()} of them on a bank line, {cells.training.mean():.0%} in the training blocks")
cells.describe().round(3)

### Step 7: Train a random forest

A random forest is a hundred decision trees, each grown on a random subset of the cells and of the features, that vote.
The share of trees voting "bank" is the bank probability. In scikit-learn that is three lines. We weight the classes so that the rare bank
cells count as much as the rest, and require at least 20 cells per leaf so the probabilities are not jumpy. About a minute.

> 🤖 This is the kind of cell an LLM writes well; scikit-learn is in its training data many thousands of times.
> Tell Gemini that the DataFrame `cells` has the feature columns listed in `features`, and 0/1 columns `is_bank` and
> `training`, and ask for a random forest trained on the rows with `training` equal to 1, with class balancing and a plot of the feature importances.
> Compare with the two cells below.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

train = cells[cells.training == 1]
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=1)
forest.fit(train[features], train.is_bank)

In [ ]:
pd.Series(forest.feature_importances_, index=features).sort_values().plot.barh(title="Feature importance");

The two curvatures carry most of the weight, as they should, but distance from the centerline comes next. The model has learned how far
from the stream banks usually are, something the curvature rule did not know.

### Step 8: Probability to cost to banks

We predict a probability for every corridor cell, including the held-out blocks, and write it back into GRASS
as a raster with `r.in.xyz`, the table-to-raster move.

In [ ]:
cells["prob"] = forest.predict_proba(cells[features])[:, 1]
xyz = cells[["x", "y", "prob"]].to_csv(index=False, header=False)
tools.r_in_xyz(input=io.StringIO(xyz), output="bank_probability", separator="comma")

In [ ]:
with gs.RegionManager(region="zoom_A"):
    prob_map = gj.Map(use_region=True, width=1000)
    prob_map.d_shade(color="bank_probability", shade="skyview_relief")
    prob_map.d_barscale(style="solid", at=[2, 5], units="meters", length=500, fontsize=14, bgcolor="none")
    prob_map.d_legend(raster="bank_probability", title="Bank probability", label_step=0.25, flags="t", fontsize=14, at=[5, 10, 60, 95])
    prob_map.show()

In reach A the probability picks out both banks as continuous bright lines.
The probability becomes cost: 100 to the power of (1 minus the probability), so a sure bank costs 1 and a sure non-bank costs 100,
the same range as in part 1 but continuous instead of two-valued. The centerline is a wall as before. The map shows the result in reach B, with the colors of the part 1 cost surface.

In [ ]:
with gs.MaskManager():
    tools.r_mask(vector="buffer")
    tools.r_mapcalc(expression=f"cost_rf = if(isnull(wall), pow(100, 1 - bank_probability), {stream_cost})")

In [ ]:
cost_colors = "0 #F2E51E\n1 #F2E51E\n100 #209F88\n100000 #482072"
tools.r_colors(map="cost_rf", rules=io.StringIO(cost_colors))

with gs.RegionManager(region="zoom_B"):
    cost_map = gj.Map(use_region=True, width=1000)
    cost_map.d_shade(color="cost_rf", shade="skyview_relief")
    cost_map.d_vect(map="start_points", size=10, icon="basic/pin", color="black", fill_color="black", legend_label="Start points")
    cost_map.d_barscale(style="solid", at=[2, 5], units="meters", length=100, fontsize=20, bgcolor="none")
    cost_map.d_legend(raster="cost_rf", title="Cost", label_values=[10, 100, 1000, 10000, 100000], flags="tl", fontsize=14, at=[5, 9, 60, 95])
    cost_map.show()

Then the same two tools as in part 1, with the new cost surface. About half a minute.

In [ ]:
tools.r_cost(flags="k", input="cost_rf", output="cumulative_cost_rf", outdir="cost_direction_rf", start_points="start_points")
tools.r_path(input="cost_direction_rf", vector_path="banks_rf", start_points="end_points")

For the comparison we need the width of the new banks, so we repeat the sampling and pairing of step 5 for them.

In [ ]:
tools.v_extract(input="banks_rf", cats="1", output="right_bank_rf")
tools.v_extract(input="banks_rf", cats="2", output="left_bank_rf")
for side in ("right_bank_rf", "left_bank_rf"):
    tools.v_to_points(input=side, output=f"{side}_points", dmax=spacing)
    tools.v_distance(from_=f"{side}_points", from_layer=2, to="stream", upload="to_along", column="along")
tools.v_db_addcolumn(map="right_bank_rf_points", layer=2, column="width double precision")
tools.v_distance(from_="right_bank_rf_points", from_layer=2, to="left_bank_rf_points", to_layer=2, upload="dist", column="width")

profile_rf = pd.DataFrame(
    tools.v_db_select(map="right_bank_rf_points", layer=2, columns="along,width", format="json")["records"]
)
profile_rf = profile_rf.apply(pd.to_numeric).sort_values("along").reset_index(drop=True) * FT_TO_M
profile_rf["along"] /= 1000

The widths of both results in one plot, with the training blocks shaded.


In [ ]:
ax = profile_axes("Width (m)")
for start in block_starts:
    ax.axvspan(start, start + block_length, color="0.92", label="training" if start == block_starts[0] else None)
ax.plot(profile.along, smooth(profile.width), lw=2, label="curvature cost")
ax.plot(profile_rf.along, smooth(profile_rf.width), lw=2, label="random forest cost")
ax.legend()

In the three shaded training blocks the two results are nearly the same: the forest learned part 1's banks there and gives them back.
Everywhere else the model had to generalize from those blocks. It agrees with part 1 wherever part 1 looked right, within a meter at
most stations, and two of the three big spikes shrink: the ones at km 4.4 and 6.3 come down from 23 and 33 m to 14 and 19 m.
Over the whole held-out part the share of stations wider than 25 m drops from 7% to under 3%. The spike at km 7.1 stays at 30 m. Let's look at the reach again, with the probability under the lines.


In [ ]:
with gs.RegionManager(grow=100, **zoom):
    rf_map = gj.Map(use_region=True, width=1000)
    rf_map.d_rast(map="skyview_relief")
    rf_map.d_vect(map="banks", width=2, color="yellow", legend_label="Banks (curvature)")
    rf_map.d_vect(map="banks_rf", width=2, color="red", legend_label="Banks (random forest)")
    rf_map.d_barscale(style="solid", at=[2, 5], units="meters", length=200, fontsize=14, bgcolor="none")
    rf_map.d_legend_vect(at=[60, 5], fontsize=14, columns=2)
    rf_map.show()

Where the curvature path had wandered out to a break in slope 20 m from the stream, the probability there is low and the new path keeps
to the inner bank: the model learned from the three training blocks how far from the centerline a bank usually is and how high it stands.
Farther down in the map both paths go out for the same stretch, the spike at km 7.1 that survived, and the probability shows no inner
bank there either. The model cannot find a bank that the terrain does not show. Whether the channel really is 30 m wide there or both
paths climbed onto a terrace is something this data cannot settle; a visit could.


Both sets of banks along the whole stream, on the web map. Zoom in where the two disagree and compare them with the hillshade and the imagery;
the plot above tells you where to look.


In [ ]:
compare_web = web_map("stream")
gj.Vector("banks", title="Banks (curvature)", style_function=lambda feature: {"color": "black"}).add_to(compare_web)
gj.Vector("banks_rf", title="Banks (random forest)", style_function=lambda feature: {"color": "magenta"}).add_to(compare_web)
folium.LayerControl().add_to(compare_web)
compare_web

One caveat. The held-out part also comes out a little narrower everywhere, by about two meters on average, including reaches where part 1
had no obvious spikes. Whether that is a correction or the model pulling the banks inward is something this data cannot tell us;
a few surveyed cross sections could.

Now look at what Gemini answered at the start of part 2. It should have warned you that a model trained on the output of a rule learns
the rule, mistakes included. That is right, and it is why the training blocks and the rest are separate: the spikes only came down because
the model never saw them. Train on everything and they stay at full height. Which blocks you train on matters as much. Move the first
block to the first two kilometers, where the channel is narrowest and part 1 is least reliable, and the km 7.1 spike comes down too,
along with everything else: the model has learned that banks sit close to the centerline. A tidier plot, from worse labels.
The real upgrade would be labels that do not come from
part 1: a few hundred meters of banks digitized by a person from the lidar, or surveyed cross sections.

> 🤖 Ask Gemini for three ways to improve the bank probability, and judge them: which could you do with the data in
> this notebook this afternoon, and which need data we do not have? Sorting its suggestions is a better use of your time than
> generating them yourself. Vetting them is still your job.

## Wrapping up

Each of the three had a different job.

- Geospatial modeling did the work. Flow routing, curvature, a cost surface and a least-cost path turned a terrain model into two bank
  lines and a channel profile, with about a dozen tools and no training data. Its weakness was one hand-tuned rule on one variable.
- Machine learning combined six variables and learned a distance-from-stream prior that the rule lacked. Tested on the 15 km of stream it had not
  seen, it removed two of the three worst spikes and left the third, and it could only do that because the spikes were kept out of
  its labels. The lesson is about the labels, and about which stretches of the stream they come from.
- The LLM explained, wrote standard code and reformatted errors. It was most useful for library code that exists in a million copies
  (scikit-learn, pandas, matplotlib) and for talking through a concept, and least useful for the parameters of a specific GRASS tool,
  for an API newer than its training data, and for anything it cannot see, such as the maps. It never said it was unsure.
  The manual, a test and your own eyes remain the way to find out whether it was right.

## Further reading

- The manuscript version of this analysis, which also shows how to assemble the dataset from the original sources, and the dataset:
  [doi:10.5281/zenodo.23600243](https://doi.org/10.5281/zenodo.23600243).
- The workflow is adapted from the USGS Remote Sensing Scorecard for montane meadows: von Nonn, J. W. and Petrasova, A. (2026),
  [doi:10.5066/P1DYFKAF](https://doi.org/10.5066/P1DYFKAF).
- Flow routing in r.watershed: Ehlschlaeger, C. R. (1989). Using the A<sup>T</sup> search algorithm to develop hydrologic models
  from digital elevation data. *Proceedings of the IGIS Symposium*, Baltimore, 275–281.
- Stream ordering addons: Jasiewicz, J. and Metz, M. (2011). A new GRASS GIS toolkit for Hortonian analysis of drainage networks.
  *Computers & Geosciences*, 37(8), 1162–1173.
- HAND: Nobre, A. D. et al. (2011). Height Above the Nearest Drainage, a hydrologically relevant new terrain model.
  *Journal of Hydrology*, 404(1–2), 13–29.
- Random forests: Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
- Curvature from a moving window: Wood, J. (1996). *The geomorphological characterisation of digital elevation models*. PhD thesis, University of Leicester.
- Sky-view factor: Zakšek, K., Oštir, K. and Kokalj, Ž. (2011). Sky-view factor as a relief visualization technique. *Remote Sensing*, 3(2), 398–415.
- GRASS tutorials, including a tutorial version of this workflow: [grass-tutorials.osgeo.org](https://grass-tutorials.osgeo.org/).